In [32]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'data' / '1_train_full.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'data' / '1_test.parquet')

## 1. 피쳐 엔지니어(파생변수 생성)

In [ ]:
# ============================================================
# 1-1. 수위 차분 파생변수 생성
# ============================================================
# wl_diff_00h = wl_t(현재수위) - wl_t_minus_00h 파생변수 5개 생성

print('train 파생변수 추가 전 크기: ', train_full.shape)
print('test 파생변수 추가 전 크기: ', test.shape)

for h in [1, 3, 6, 12, 24]:
    train_full[f"wl_diff_{h}h"] = (
        train_full['wl_t'] - train_full[f"wl_t_minus_{h}h"]
    )
    test[f"wl_diff_{h}h"] = (
        test['wl_t'] - test[f"wl_t_minus_{h}h"]
    )

print('train 수위차분 파생변수 추가 후 크기: ', train_full.shape)
print('test 수위차분 파생변수 추가 후 크기: ', test.shape)


train 파생변수 추가 전 크기:  (749657, 25)
test 파생변수 추가 전 크기:  (186263, 24)
train 수위차분 파생변수 추가 후 크기:  (749657, 30)
test 수위차분 파생변수 추가 후 크기:  (186263, 29)


In [ ]:
# ============================================================
# 1-2. 강우강도 파생변수 생성
# ============================================================
# imerg_int_0to0h = (imerg_rain_00h - imerg_rain_00h)/hours
# sfc_int_0to0h = (sfc_rain_00h - sfc_rain_00h)/hours

print('train 파생변수 추가 전 크기: ', train_full.shape)
print('test 파생변수 추가 전 크기: ', test.shape)


for df in [train_full, test]:

    df['imerg_int_1to6h'] = (df['imerg_rain_6h'] - df['imerg_rain_1h']) / 5
    df['imerg_int_6to24h'] = (df['imerg_rain_24h'] - df['imerg_rain_6h']) / 18

    df['sfc_int_1to6h'] = (df['sfc_rain_6h'] - df['sfc_rain_1h']) / 5
    df['sfc_int_6to24h'] = (df['sfc_rain_24h'] - df['sfc_rain_6h']) / 18


print('train 강우강도 파생변수 추가 후 크기: ', train_full.shape)
print('test 강우강도 파생변수 추가 후 크기: ', test.shape)

train 파생변수 추가 전 크기:  (749657, 30)
test 파생변수 추가 전 크기:  (186263, 29)
train 강우강도 파생변수 추가 후 크기:  (749657, 34)
test 강우강도 파생변수 추가 후 크기:  (186263, 33)


In [ ]:
# ============================================================
# 1-3. 최종 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 독립변수군
feature_cols = [
    col for col in train_full.columns
    if col not in ('timestamp', target_col)
]

assert len(feature_cols) == 32, (
    f'독립변수 개수 이상: 예상 32개, 현재 {len(feature_cols)}개'
)

## 2. 이상치 제거

In [ ]:
# ==============================================================
# 2-1. 이상치(계측오류) 제거: train_full 종속변수 이상치 제거(1개)
# ==============================================================
# 변수별 분포특성 확인단계에서 확인됨

# 제거 조건 설정
cond_maxrise_outlier = train_full['target_maxrise_6h'] < -8  # 이거 자체가 불리언 조건문임
print(f" cond_maxrise_outlier 조건에 해당하는 데이터 개수: {cond_maxrise_outlier.sum()}")  # 불리언조건문은 sum으로 개수를 셀 수 있음
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 

# 제거 대상 지정
print()
maxrise_outlier_row_id = (
    train_full
    .loc[cond_maxrise_outlier]
    .index
)
print(maxrise_outlier_row_id)

# 이상치 제거
train_full = train_full.drop(index = maxrise_outlier_row_id)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")
print()

 cond_maxrise_outlier 조건에 해당하는 데이터 개수: 1
이상치 행 제거 전 데이터 개수: 749657

Index(['TR_000583385'], dtype='object', name='row_id')
이상치 행 제거 후 train셋 행 개수: 749656



In [ ]:
# ==============================================================
# 2-2. 이상치(계측오류) 제거: train_full 수위변수 이상치 제거(32개)
# ==============================================================
# 변수간 상관계수 분석에서 확인됨

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 제거 조건 설정
cond_wl_out_ST1 = train_full['station_id'] == 'ST0445'
cond_wl_out_ST2 = train_full['station_id'] == 'ST0066'
cond_wl_out_ST0445 = (train_full[wl_cols] > 15).any(axis=1)
cond_wl_out_ST0066 = (train_full[wl_cols] > 23).any(axis=1)
cond_wl_f = (
    (cond_wl_out_ST1 & cond_wl_out_ST0445)|
    (cond_wl_out_ST2 & cond_wl_out_ST0066)
    )

print(
    f"ccond_wl_f 조건에 해당하는 데이터 개수: "
    f"{cond_wl_f.sum()}"
)  # 불리언조건문은 sum으로 개수를 셀 수 있음
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 

# 제거 대상 지정
wl_outlier_row_id = (
    train_full
    .loc[cond_wl_f]
    .index
)

# 이상치 제거
train_full = train_full.drop(index = wl_outlier_row_id)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")
print()

ccond_wl_f 조건에 해당하는 데이터 개수: 32
이상치 행 제거 전 데이터 개수: 749656
이상치 행 제거 후 train셋 행 개수: 749624



In [ ]:
# ==============================================================
# 2-3. 이상치(계측오류) 제거: train_full 수위변수 이상치 제거(15개)
# ==============================================================
# train, test셋 분포 비교에서 확인

# 제거 조건 설정
cond_wl_out_ST3 = train_full['station_id'] == 'ST0089'
cond_wl_out_ST4 = train_full['station_id'] == 'ST0158'
cond_wl_out_ST5 = train_full['station_id'] == 'ST0448'
cond_wl_out_ST6 = train_full['station_id'] == 'ST0007'
cond_wl_out_ST0089 = (train_full[wl_cols] > 6).any(axis=1)
cond_wl_out_ST0158 = (train_full[wl_cols] > 7).any(axis=1)
cond_wl_out_ST0448 = (
    (train_full[wl_cols] > 9).any(axis=1) &
    (train_full.index != 'TR_000199803')
)
cond_wl_out_ST0007 = (train_full[wl_cols] > 8).any(axis=1)
cond_wl_f2 = (
    (cond_wl_out_ST3 & cond_wl_out_ST0089)|
    (cond_wl_out_ST4 & cond_wl_out_ST0158)|
    (cond_wl_out_ST5 & cond_wl_out_ST0448)|
    (cond_wl_out_ST6 & cond_wl_out_ST0007)
    )

print(
    f"ccond_wl_f 조건에 해당하는 데이터 개수: "
    f"{cond_wl_f2.sum()}"
)  # 불리언조건문은 sum으로 개수를 셀 수 있음
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 

# 제거 대상 지정
wl_outlier2_row_id = (
    train_full
    .loc[cond_wl_f2]
    .index
)

# 이상치 제거
train_full = train_full.drop(index = wl_outlier2_row_id)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")
print()


ccond_wl_f 조건에 해당하는 데이터 개수: 15
이상치 행 제거 전 데이터 개수: 749624
이상치 행 제거 후 train셋 행 개수: 749609



In [9]:
# ============================================================
# 2-4. 최종 train_full, test 저장
# ============================================================

train_full.to_parquet(PROJECT_ROOT / 'data' / '2_train_full_fe.parquet')
test.to_parquet(PROJECT_ROOT / 'data' / '2_test_fe.parquet')

## 3. 강우사상 라벨링, GroupKFold 5-Fold 배정(CSV로 저장)

In [ ]:
# 여기서부터 실행할 때를 위해 작성
# from pathlib import Path
# PROJECT_ROOT = Path.cwd().parent

# import pandas as pd

# # ============================================================
# # 0. 데이터 불러오기
# # ============================================================

# train_full = pd.read_parquet(PROJECT_ROOT / 'data' / '2_train_full_fe.parquet')
# test = pd.read_parquet(PROJECT_ROOT / 'data' / '2_test_fe.parquet')

In [39]:
# ============================================================
# 3-1. 강우사상별 데이터 분할 -> event_id 컬럼 생성
# ============================================================

# 전체 timestamp 추출(중복없이), 시간순 정렬
event_time = (
    train_full[['timestamp']] 
    .drop_duplicates() 
    .sort_values('timestamp') 
    .reset_index(drop = True) 
)

# 차분: diff 계산
event_time['event_time_diff'] = (
    event_time['timestamp']
    .diff()
)

# 차분값이 1시간이 아닌 경우를 event 경계로 추출
cond_split_time = (
    event_time['event_time_diff'] != pd.Timedelta(hours=1)
)

# 강우사상 시간대별 event_id 부여
event_time['event_id'] = cond_split_time.cumsum()

# 시험처가 제시한 강우사상 개수와 맞는지 확인: 122개
assert event_time['event_id'].nunique() == 122, (
    f"주최측이 제시한 강우사상 122개, 현재 {event_time['event_id'].nunique()}개, 오류"
)

# train_full에 event_id 컬럼 붙이기(매핑)
rule_event_id_map = event_time.set_index('timestamp')['event_id']  # 매핑규칙(조인대상 인덱스 필수)

train_full['event_id'] = train_full['timestamp'].map(rule_event_id_map)

assert not train_full['event_id'].isnull().any(), (
    'event_id 매핑 누락 발생'
)

In [13]:
# ============================================================
# 3-2. GroupKFold 5-Fold 배정, csv 저장
# ============================================================

from sklearn.model_selection import GroupKFold

# ------------------------------------------------------------
# Group 설정: 강우사상(event_id) 단위로 fold 배정
# ------------------------------------------------------------

kf_groups = train_full['event_id']

print('<시간사상 Group 확인>')
print(f'전체 학습 행 수   : {len(train_full):,}')
print(f'시간사상 개수     : {kf_groups.nunique():,}')
print()

assert not kf_groups.isnull().any(), 'event_id 결측치 존재'


# ------------------------------------------------------------
# GroupKFold 정의
# ------------------------------------------------------------

N_SPLITS = 5

gkf = GroupKFold(n_splits=N_SPLITS)
# shuffle=False(기본값), Fold별 행 개수가 비슷하게 배정

# 시간사상별 Fold 번호 저장용 데이터프레임
kf_fold_f = pd.DataFrame({
    'event_id': kf_groups.unique()
})

kf_fold_f['fold_num'] = 0  # Fold 번호 컬럼 생성


# ------------------------------------------------------------
# Fold 배정 및 검증
# ------------------------------------------------------------

for fold, (train_idx, valid_idx) in enumerate(  # 반환값: fold 번호, (해당폴드 train 인덱스, val 인덱스)
    gkf.split(train_full, groups=kf_groups),  # 5개 fold 배정 결정
    start=1  # fold 1부터 시작
):
    # Fold별 train, val 사상 중복 여부 확인
    train_groups = set(kf_groups.iloc[train_idx])
    valid_groups = set(kf_groups.iloc[valid_idx])
    overlap = train_groups & valid_groups
    assert len(overlap) == 0, f'Fold {fold}: train, val 간에 사상 중복'

    # 검증 Fold에 해당하는 시간사상에 Fold 번호 부여
    kf_fold_f.loc[
        kf_fold_f['event_id'].isin(valid_groups),  # val그룹에 속하는 event_id는
        'fold_num'                                 # fold_num 컬럼에
    ] = fold                                       # 현재 fold 번호를 기재

    print(
        f'Fold {fold} | '
        f'train={len(train_idx):,} | '  # 인덱스 개수(행 개수)
        f'valid={len(valid_idx):,} | '
        f'train events={len(train_groups):,} | '  # event_id 개수
        f'valid events={len(valid_groups):,} | '
        f'event overlap={len(overlap)}'
    )


# ------------------------------------------------------------
# 3-4. Fold 배정 결과 최종 검증
# ------------------------------------------------------------

assert kf_fold_f['event_id'].is_unique, 'event_id 중복'
assert kf_fold_f['fold_num'].between(1, N_SPLITS).all(), 'Fold 배정 누락'


# ------------------------------------------------------------
# 3-5. Fold 배정 CSV 저장
# ------------------------------------------------------------

kf_fold_f.to_csv(
    PROJECT_ROOT / 'data' / '0_kf_train_full_match.csv',
    index=False
)

print('\nFold 배정 CSV 저장 완료')


<시간사상 Group 확인>
전체 학습 행 수   : 749,609
시간사상 개수     : 122

Fold 1 | train=599,625 | valid=149,984 | train events=100 | valid events=22 | event overlap=0
Fold 2 | train=599,624 | valid=149,985 | train events=97 | valid events=25 | event overlap=0
Fold 3 | train=599,673 | valid=149,936 | train events=97 | valid events=25 | event overlap=0
Fold 4 | train=599,830 | valid=149,779 | train events=97 | valid events=25 | event overlap=0
Fold 5 | train=599,684 | valid=149,925 | train events=97 | valid events=25 | event overlap=0

Fold 배정 CSV 저장 완료


## 4. Tidal, Normal 데이터 분할

In [ ]:
# ============================================================
# 4-1. Tidal 데이터 필터링
# ============================================================

# 조건 설정
tidal_cond = (
    (train_full['wl_diff_6h'] < -1) &  # 수위하강이 발생하면서(0이 아닌)
    ((train_full[target_col] + train_full['wl_diff_6h']).abs() < 2)  # 수위회복 패턴이 뒤이어 발생하는 조건
)

# 원본 station_id 결측치 검증
assert not train_full['station_id'].isnull().any(), "station_id 결측치 존재"

tidal_station_df = pd.DataFrame({
    'total_count' : train_full['station_id'].value_counts(),
    'tidal_count' : train_full.loc[tidal_cond, 'station_id'].value_counts()
}).fillna(0).astype(int)

# 한번 더 검증
assert not tidal_station_df.isnull().any().any(), "아직 NaN 있음"

# 관측소별 tidal 비율 추가
tidal_station_df['tidal_ratio(%)'] = (
    tidal_station_df['tidal_count'] / tidal_station_df['total_count'] * 100
)

# tidal_count기준 내림차순 정렬
tidal_station_df = tidal_station_df.sort_values(
    'tidal_count', ascending=False
)

tidal_stations = (
    tidal_station_df
    .head(9)  # 기존 Train(Validation 제외)에서 Tidal 발생 비율 10% 초과였던 9개 관측소 유지
    .index
)

print(tidal_stations)

Index(['ST0326', 'ST0358', 'ST0292', 'ST0347', 'ST0159', 'ST0110', 'ST0415',
       'ST0313', 'ST0234'],
      dtype='object', name='station_id')


In [41]:
# ============================================================
# 4-2. Tidal / Normal 데이터 분리
# ============================================================

# 4-1. Train 분리
tidal_train = train_full.loc[
    train_full['station_id'].isin(tidal_stations)
].copy()

normal_train = train_full.loc[
    ~train_full['station_id'].isin(tidal_stations)
].copy()

# 4-2. Test 분리
tidal_test = test.loc[
    test['station_id'].isin(tidal_stations)
].copy()

normal_test = test.loc[
    ~test['station_id'].isin(tidal_stations)
].copy()

# 4-3. 분리 결과 확인
print('<Train 분리 결과>')
print(f'Train 전체 : {len(train_full):,}')
print(f'Tidal      : {len(tidal_train):,}')
print(f'Normal     : {len(normal_train):,}')

print('\n<Test 분리 결과>')
print(f'Test 전체  : {len(test):,}')
print(f'Tidal      : {len(tidal_test):,}')
print(f'Normal     : {len(normal_test):,}')

# 4-4. 데이터 분리 검증
assert len(tidal_train) + len(normal_train) == len(train_full)
assert len(tidal_test) + len(normal_test) == len(test)

print('\n데이터 분리 검증 완료')

<Train 분리 결과>
Train 전체 : 749,609
Tidal      : 14,251
Normal     : 735,358

<Test 분리 결과>
Test 전체  : 186,263
Tidal      : 3,573
Normal     : 182,690

데이터 분리 검증 완료


## 5. 독립변수 / 종속변수 설정

In [42]:
# ============================================================
# 5. 독립변수 / 종속변수 분리
# ============================================================

assert len(feature_cols) == 32, (
    f'독립변수 개수 이상: 예상 32개, 현재 {len(feature_cols)}개'
)

# Tidal
X_tidal_train = tidal_train[feature_cols].copy()
X_tidal_test = tidal_test[feature_cols].copy()

y_tidal_train = tidal_train[target_col]

# Normal
X_normal_train = normal_train[feature_cols].copy()
X_normal_test = normal_test[feature_cols].copy()

y_normal_train = normal_train[target_col]

# 데이터 분리 검증
assert X_tidal_train.index.equals(y_tidal_train.index)
assert X_normal_train.index.equals(y_normal_train.index)

assert X_tidal_train.columns.equals(X_tidal_test.columns)
assert X_normal_train.columns.equals(X_normal_test.columns)

print('독립변수 / 종속변수 분리 검증 완료')

독립변수 / 종속변수 분리 검증 완료


## 6. Tidal 데이터셋 모델 생성: 단일 홀드아웃, CatBoost, ExtraTree 앙상블

In [ ]:
# ============================================================
# 6-1. CatBoost에 넣을 범주형 변수 처리
# ============================================================

for X in [
    X_tidal_train, X_tidal_test,
    X_normal_train, X_normal_test
]:
    X['station_id'] = X['station_id'].astype(str)

In [18]:
# ============================================================
# 6-2. CatBoost 모델로 tidal 예측
# ============================================================
from catboost import CatBoostRegressor

# Tidal CatBoost 모델

cat_tidal = CatBoostRegressor(
    iterations=2976,  # train tidal 5-Fold best rmse의 best iteration
    random_state=42,
    cat_features=['station_id'],
    verbose=200
)

cat_tidal.fit(
    X_tidal_train, y_tidal_train,
)

pred_cat_tidal = cat_tidal.predict(X_tidal_test)

# Tidal CatBoost 모델 저장
cat_tidal.save_model(
    str(PROJECT_ROOT / 'model' / 'cat_tidal.cbm')
)


Learning rate set to 0.02567
0:	learn: 1.7277226	total: 125ms	remaining: 6m 11s
200:	learn: 0.4373586	total: 5.7s	remaining: 1m 18s
400:	learn: 0.3813116	total: 11.5s	remaining: 1m 13s
600:	learn: 0.3465941	total: 17s	remaining: 1m 7s
800:	learn: 0.3223825	total: 22.6s	remaining: 1m 1s
1000:	learn: 0.3045066	total: 28.2s	remaining: 55.7s
1200:	learn: 0.2902351	total: 34.1s	remaining: 50.4s
1400:	learn: 0.2786815	total: 39.8s	remaining: 44.7s
1600:	learn: 0.2678048	total: 45.5s	remaining: 39.1s
1800:	learn: 0.2581860	total: 51.3s	remaining: 33.5s
2000:	learn: 0.2495655	total: 57.1s	remaining: 27.8s
2200:	learn: 0.2416716	total: 1m 2s	remaining: 22.1s
2400:	learn: 0.2345677	total: 1m 8s	remaining: 16.4s
2600:	learn: 0.2279707	total: 1m 15s	remaining: 10.9s
2800:	learn: 0.2220487	total: 1m 21s	remaining: 5.07s
2975:	learn: 0.2172739	total: 1m 26s	remaining: 0us


In [ ]:
# ============================================================
# 6-3. ExtraTrees 모델에 넣을 범주형 변수 처리: OneHotEncoding
# ============================================================

from sklearn.preprocessing import OneHotEncoder

# OneHotEncoder 정의 및 학습
ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

ohe.fit(X_tidal_train[['station_id']])

# Train / Test 원핫인코딩
X_tidal_train_ohe = pd.concat([
    X_tidal_train.drop(columns='station_id'),  # 기존에 있던 'station_id' 컬럼은 제거
    pd.DataFrame(  # 원핫인코딩 결과 데이터프레임으로 만들어서 붙이기
        ohe.transform(X_tidal_train[['station_id']]),  
        columns=ohe.get_feature_names_out(),  # 컬럼 이름을 해당 station_id로 받아오기
        index=X_tidal_train.index
    )
], axis=1)

X_tidal_test_ohe = pd.concat([
    X_tidal_test.drop(columns='station_id'),
    pd.DataFrame(
        ohe.transform(X_tidal_test[['station_id']]),
        columns=ohe.get_feature_names_out(),
        index=X_tidal_test.index
    )
], axis=1)

# 검증
assert 'station_id' not in X_tidal_train_ohe.columns
assert X_tidal_train_ohe.columns.equals(X_tidal_test_ohe.columns)
assert X_tidal_train_ohe.index.equals(y_tidal_train.index)

# 원핫인코딩 전후 Shape 비교
print(f'Tidal Train : {X_tidal_train.shape} -> {X_tidal_train_ohe.shape}')
print(f'Tidal Test  : {X_tidal_test.shape} -> {X_tidal_test_ohe.shape}')

# 학습된 OneHotEncoder 저장
import joblib

joblib.dump(
    ohe,
    PROJECT_ROOT / 'model' / 'ohe_tidal_station_id.joblib'
)

Tidal Train : (14251, 32) -> (14251, 40)
Tidal Test  : (3573, 32) -> (3573, 40)


['c:\\Users\\soohok402\\내 드라이브\\Github\\AI_Data_exhibition\\model\\ohe_tidal_station_id.joblib']

In [20]:
# ============================================================
# 6-4. ExtraTrees 모델로 tidal 예측
# ============================================================
from sklearn.ensemble import ExtraTreesRegressor

# Tidal ExtraTrees 모델
et_tidal = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1,
    verbose=0
        )


et_tidal.fit(
    X_tidal_train_ohe, y_tidal_train,
)

pred_et_tidal = et_tidal.predict(X_tidal_test_ohe)

# Tidal ExtraTrees 모델 저장
import joblib

joblib.dump(
    et_tidal,
    PROJECT_ROOT / 'model' / 'et_tidal_ExtraTrees.joblib'
)

['c:\\Users\\soohok402\\내 드라이브\\Github\\AI_Data_exhibition\\model\\et_tidal_ExtraTrees.joblib']

In [21]:
# ============================================================
# 6-5. CatBoost 45% + ExtraTrees 55% 앙상블 예측
# ============================================================

pred_ensemble_tidal = pd.Series(
    pred_cat_tidal * 0.45 + pred_et_tidal * 0.55,
    index=X_tidal_test.index
)

# 앙상블 예측 결과 검증
assert pred_ensemble_tidal.index.equals(X_tidal_test.index)
assert len(pred_ensemble_tidal) == len(tidal_test)
assert pred_ensemble_tidal.notna().all()

print(f'Tidal Test: {len(tidal_test):,}행')
print(f'앙상블 예측: {len(pred_ensemble_tidal):,}행')
print('Tidal 앙상블 예측 검증 완료')

Tidal Test: 3,573행
앙상블 예측: 3,573행
Tidal 앙상블 예측 검증 완료


## 7. Normal 데이터셋 모델 생성: K_Fold 교차검증, CatBoost

In [ ]:
# ============================================================
# 7-1. 고정 Fold CSV 불러오기 및 Train 데이터 매칭
# ============================================================

# 고정 Fold 매핑 파일 불러오기
fold_match = pd.read_csv(
    PROJECT_ROOT / 'data' / '0_kf_train_full_match.csv'
)

# event_id 기준으로 Fold 번호 매칭
normal_train['fold_num'] = normal_train['event_id'].map(
    fold_match.set_index('event_id')['fold_num']  # 매핑규칙: 'event_id'로 조인, 'fold_num' 부여
)

# 매칭 검증
assert normal_train['fold_num'].notna().all()
assert normal_train['fold_num'].between(1, N_SPLITS).all()

print('Normal Fold 매칭 완료')

Normal Fold 매칭 완료


In [24]:
# ============================================================
# 7-2. Normal CatBoost 5-Fold 학습 및 예측
# ============================================================

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error
import numpy as np

# 결과 저장용 변수
normal_oof_pred = pd.Series(
    index=X_normal_train.index,
    dtype=float
)

normal_test_preds = pd.DataFrame(
    index=X_normal_test.index
)

normal_fold_results = []

# Fold별 학습
for fold in range(1, N_SPLITS + 1):

    print(f'\n========== Normal Fold {fold} ==========')

    # 기존 Fold 번호로 분할
    train_mask = normal_train['fold_num'] != fold
    valid_mask = normal_train['fold_num'] == fold

    X_kf_tr = X_normal_train.loc[train_mask]
    X_kf_val = X_normal_train.loc[valid_mask]

    y_kf_tr = y_normal_train.loc[train_mask]
    y_kf_val = y_normal_train.loc[valid_mask]

    assert len(X_kf_tr) > 0 and len(X_kf_val) > 0, f'Fold {fold}: 학습/검증 데이터 없음'

    # CatBoost 최종 선정모델
    model = CatBoostRegressor(
        iterations=1100,  # train 5-Fold에서 fold별 best iteration 편차가 큼 -> 평균 선택
        random_seed=42,
        cat_features=['station_id'],
        thread_count=-1
    )

    model.fit(
        X_kf_tr, y_kf_tr,
        verbose=500
    )

    # Fold별 kf_val 예측
    val_pred = model.predict(X_kf_val)

    # Fold별 OOF 예측 및 RMSE
    normal_oof_pred.loc[X_kf_val.index] = val_pred
    fold_oof_rmse = root_mean_squared_error(y_kf_val, val_pred)

    # Fold별 Test 전체 예측
    test_pred = model.predict(X_normal_test)
    normal_test_preds[f'fold_{fold}'] = test_pred  # Fold별 예측결과가 하나의 컬럼으로 삽입
  
    # Fold 결과 저장
    normal_fold_results.append({
        'fold': fold,
        'train_count': len(X_kf_tr),
        'valid_count': len(X_kf_val),
        'oof_rmse': fold_oof_rmse,
    })

    print(
        f'Fold {fold} | '
        f'OOF RMSE={fold_oof_rmse:.6f} | '
    )

    # Normal CatBoost 모델 저장
    model.save_model(
        str(PROJECT_ROOT / 'model' / f"cat_{fold}fold_normal.cbm")
    )


========== Normal Fold 1 ==========
Learning rate set to 0.103781
0:	learn: 0.2710637	total: 103ms	remaining: 1m 52s
500:	learn: 0.1898264	total: 42.2s	remaining: 50.5s
1000:	learn: 0.1776401	total: 1m 28s	remaining: 8.74s
1099:	learn: 0.1759080	total: 1m 38s	remaining: 0us
Fold 1 | OOF RMSE=0.233810 | 

========== Normal Fold 2 ==========
Learning rate set to 0.10378
0:	learn: 0.2881390	total: 102ms	remaining: 1m 52s
500:	learn: 0.2031983	total: 48.9s	remaining: 58.5s
1000:	learn: 0.1896214	total: 1m 37s	remaining: 9.6s
1099:	learn: 0.1877447	total: 1m 45s	remaining: 0us
Fold 2 | OOF RMSE=0.165635 | 

========== Normal Fold 3 ==========
Learning rate set to 0.103782
0:	learn: 0.2553372	total: 104ms	remaining: 1m 54s
500:	learn: 0.1814054	total: 46.3s	remaining: 55.3s
1000:	learn: 0.1691034	total: 1m 34s	remaining: 9.37s
1099:	learn: 0.1672998	total: 1m 43s	remaining: 0us
Fold 3 | OOF RMSE=0.260627 | 

========== Normal Fold 4 ==========
Learning rate set to 0.103785
0:	learn: 0.27720

In [ ]:
# ============================================================
# 7-3. Normal CatBoost 학습 및 예측 결과 검증
# ============================================================

# OOF 예측 검증
assert normal_oof_pred.index.equals(y_normal_train.index)
assert normal_oof_pred.notna().all()

# Test 예측 검증
assert normal_test_preds.index.equals(X_normal_test.index)
assert normal_test_preds.shape == (len(X_normal_test), N_SPLITS)
assert normal_test_preds.notna().all().all()

# 전체 OOF RMSE
normal_oof_rmse = root_mean_squared_error(
    y_normal_train, normal_oof_pred
)

print(f'Normal 전체 OOF RMSE: {normal_oof_rmse:.6f}')
print(f'Normal Test 예측: {normal_test_preds.shape}')
print('Normal CatBoost 검증 완료')

Normal 전체 OOF RMSE: 0.217860
Normal Test 예측: (182690, 5)
Normal CatBoost 검증 완료


In [ ]:
# ============================================================
# 7-4. Normal CatBoost 5-Fold 개별모델 앙상블 예측
# ============================================================

pred_ensemble_normal = normal_test_preds.mean(axis=1)  # axis=1은 행별집계
# 결과물: 하나의 행에 fold별 예측값이 컬럼별로 배치 -> 행별집계 -> 행별 1개 mean값(인덱스 유지)

# 검증
assert pred_ensemble_normal.index.equals(X_normal_test.index)
assert pred_ensemble_normal.notna().all()

print(f'Normal Test: {len(normal_test):,}행')
print(f'앙상블 예측: {len(pred_ensemble_normal):,}행')
print('Normal 앙상블 예측 검증 완료')

Normal Test: 182,690행
앙상블 예측: 182,690행
Normal 앙상블 예측 검증 완료


## 8. Tidal / Normal 예측 결과 통합 및 제출파일 생성

In [27]:
# ============================================================
# 8. Tidal / Normal 예측 결과 통합 및 제출파일 생성
# ============================================================

# 예측 결과 통합
pred_test = pd.concat([
    pred_ensemble_tidal,
    pred_ensemble_normal
])

# 제출용 데이터프레임 생성
final_result = pred_test.rename('pred_maxrise_6h').reset_index()

# 검증
assert final_result.columns.tolist() == ['row_id', 'pred_maxrise_6h']
assert len(final_result) == len(test)
assert final_result['row_id'].is_unique
assert set(final_result['row_id']) == set(test.index)
assert final_result['pred_maxrise_6h'].notna().all()

print('제출 데이터 생성 및 검증 완료')
print(final_result.shape)

제출 데이터 생성 및 검증 완료
(186263, 2)


In [30]:
final_result.to_csv(
    PROJECT_ROOT / 'data' / 'result_v1.csv',
    index=False,
    encoding='utf-8'
)

In [28]:
final_result.head(100)

,row_id,pred_maxrise_6h
0,TE_000063287,-0.212625
1,TE_000067268,2.738044
2,TE_000049674,0.132557
3,TE_000038339,-0.231214
4,TE_000064879,2.713916
...,...,...
95,TE_000145092,-0.841638
96,TE_000134399,-0.937899
97,TE_000044584,0.464997
98,TE_000128178,1.115125
